In [1]:
CSV_PATH = "data/solusd_5m_coinbase.csv"

import os, sys, pandas as pd, numpy as np

# Make sure we can import your project modules when the notebook sits in backend/
if "." not in sys.path: sys.path.append(".")
if os.path.abspath(".").endswith(os.sep + "backend"):
    # Running from backend/ directory
    pass
else:
    # If not, try to find backend/ by walking up
    here = os.path.abspath(".")
    parts = here.split(os.sep)
    if "backend" in parts:
        idx = parts.index("backend")
        proj_root = os.sep.join(parts[:idx+1])
        os.chdir(proj_root)

print("CWD:", os.getcwd())
print("CSV:", os.path.abspath(CSV_PATH))

CWD: /Users/dongillee/lms/backend
CSV: /Users/dongillee/lms/backend/data/solusd_5m_coinbase.csv


In [2]:
# Load CSV
df = pd.read_csv(CSV_PATH)
# Expect columns: ts, open, high, low, close[, volume]
if "ts" not in df.columns:
    raise ValueError("CSV missing 'ts' column.")
# Parse timestamps to UTC-aware
df["ts"] = pd.to_datetime(df["ts"], utc=True, errors="coerce")
df = df.dropna(subset=["ts"]).sort_values("ts").reset_index(drop=True)

# Keep only necessary columns in standard names
need = ["ts","open","high","low","close"]
missing = [c for c in need if c not in df.columns]
if missing:
    raise ValueError(f"CSV missing columns: {missing}")

view = df[need].copy()
view.tail(3)

,ts,open,high,low,close
450371,2025-10-03 10:10:00+00:00,228.78,229.12,228.28,228.62
450372,2025-10-03 10:15:00+00:00,228.56,228.57,227.26,227.76
450373,2025-10-03 10:20:00+00:00,227.71,228.77,227.35,228.49


In [3]:
from accounts.analysis_helpers import resample_ohlc
from accounts.strategies.signals import isKalmanUptrend, isKalmanBuy, isKalmanSell # helper from signals.py
from accounts.strategies.signals import is_hh_hl, isLorentzianBuy, isLorentzianSell
from accounts.strategies.signals import isKalman_regression_down_break
from accounts.strategies.test_signals import long_tester
#from accounts.strategies.lorentzian import lorentzian_positions_advta as lc


df = resample_ohlc(view, "1h")

In [ ]:


trades, e_mask, x_mask = long_tester(
    df,
    use_lorentzian_buy=False,
    use_kalman_buy=True,
    require_HH=False,
    require_HL=False,
    hhhl_func=is_hh_hl,   # <— add this OR precompute columns
)


In [ ]:
trades

In [4]:
import os, re, glob
import pandas as pd


def _load_and_standardize_stock_csv(path: str) -> pd.DataFrame:
    raw = pd.read_csv(path)
    raw.columns = [c.strip() for c in raw.columns]

    def _money(s: pd.Series) -> pd.Series:
        return pd.to_numeric(s.astype(str).str.replace(r"[$,]", "", regex=True), errors="coerce")
    def _intlike(s: pd.Series) -> pd.Series:
        return pd.to_numeric(s.astype(str).str.replace(",", "", regex=True), errors="coerce")

    df = pd.DataFrame({
        "ts":     pd.to_datetime(raw["Date"], errors="coerce"),
        "open":   _money(raw["Open"]),
        "high":   _money(raw["High"]),
        "low":    _money(raw["Low"]),
        "close":  _money(raw["Close/Last"]),
        "volume": _intlike(raw["Volume"]),
    }).dropna(subset=["ts","close"]).sort_values("ts").reset_index(drop=True)

    return df

def _extract_ticker(filename: str) -> str:
    base = os.path.basename(filename)
    m = re.search(r"HistoricalData_([A-Za-z]{3,5})\.csv$", base)
    if m: return m.group(1).upper()
    m = re.search(r"HistoricalData_(.+)\.csv$", base)
    return m.group(1).upper() if m else base

def run_batch_long_tests(
    data_dir: str = "backend/data/stocks",
    *,
    use_lorentzian_buy=False,
    use_kalman_buy=True,
    require_HH=False,
    require_HL=False,
    tolerance=2,
    fee=0.002,
) -> dict[str, pd.DataFrame]:
    # ✅ make path absolute so it works regardless of current working dir
    data_dir = os.path.abspath(os.path.expanduser(data_dir))
    pattern = os.path.join(data_dir, "HistoricalData_*.csv")
    files = sorted(glob.glob(pattern))

    # quick sanity log (delete if you want it silent)
    print(f"[INFO] CWD={os.getcwd()}")
    print(f"[INFO] Looking for: {pattern}")
    print(f"[INFO] Found {len(files)} files")

    results: dict[str, pd.DataFrame] = {}
    for path in files:
        tkr = _extract_ticker(path)
        try:
            df = _load_and_standardize_stock_csv(path)
            trades, _, _ = long_tester(
                df,
                use_lorentzian_buy=use_lorentzian_buy,
                use_kalman_buy=use_kalman_buy,
                require_HH=require_HH,
                require_HL=require_HL,
                tolerance=tolerance,
                fee=fee,
            )
            results[tkr] = trades  # even if empty, you’ll still see the key
            print(f"[OK] {tkr}: {len(trades)} trades")
        except Exception as e:
            print(f"[FAIL] {tkr}: {e}")
            results[tkr] = pd.DataFrame()

    return results



In [ ]:
tables = run_batch_long_tests(
    data_dir="data/stocks",
    use_lorentzian_buy=True,
    use_kalman_buy=False,
    require_HH=False,
    require_HL=True,
    tolerance=2,
    fee=0.002,
)

print({tkr: len(tbl) for tkr, tbl in tables.items()})

In [5]:
ABCL = _load_and_standardize_stock_csv("data/stocks/HistoricalData_ABCL.csv")

In [6]:
ABCL["isKRegDownBreak"] = isKalman_regression_down_break(ABCL) 

In [7]:
ABCL.to_csv("ABCL_isKRegDownBreak.csv")